In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#api fetching lib
from tqdm import tqdm
import requests
import time
import os
from concurrent.futures import ThreadPoolExecutor
import seaborn as sns

In [2]:
df = pd.read_csv('train_data.csv')

In [3]:
df

,QueryID,ResponseID,QueryName,ResponseName,ReleaseDate,RequiredAge,DemoCount,DeveloperCount,DLCCount,Metacritic,...,LegalNotice,Reviews,SupportedLanguages,Website,PCMinReqsText,PCRecReqsText,LinuxMinReqsText,LinuxRecReqsText,MacMinReqsText,MacRecReqsText
0,10,10,Counter-Strike,Counter-Strike,Nov 1 2000,0,0,1,0,88,...,,,English French German Italian Spanish Simplifi...,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
1,20,20,Team Fortress Classic,Team Fortress Classic,Apr 1 1999,0,0,1,0,0,...,,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
2,30,30,Day of Defeat,Day of Defeat,May 1 2003,0,0,1,0,79,...,,,English French German Italian Spanish,http://www.dayofdefeat.com/,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
3,40,40,Deathmatch Classic,Deathmatch Classic,Jun 1 2001,0,0,1,0,0,...,,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
4,50,50,Half-Life: Opposing Force,Half-Life: Opposing Force,Nov 1 1999,0,0,1,0,0,...,,,English French German Korean,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11352,567020,567020,TILE,TILE,Dec 9 2016,0,0,1,0,0,...,,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) Vista / 7P...,,,,,
11353,567660,567660,Baseball Riot,Baseball Riot,Jan 17 2017,0,0,1,0,0,...,Copyright (c) 2016 10tons Ltd.,,English**languages with full audio support,http://www.10tons.com/Game/baseball_riot.html,Minimum:OS: Windows XP / Vista / 7 / 8 / 10Pro...,,,,,
11354,567860,567860,Passage 4,Passage 4,Dec 13 2016,0,0,1,0,0,...,2016 copyright by netmin e.K.,,English* French Italian German* Spanish Dutch*...,http://www.libredia.com,Minimum:OS: Windows 2000/XP/Vista/7/8/10Proces...,,,,,
11355,567940,567940,Piximalism,Piximalism,Sep 26 2019,0,0,1,0,0,...,,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) XP / Vista...,Recommended:OS: Microsoft(r) Windows(r) XP / V...,,,,


In [ ]:
#fetching function
def get_top_tags(app_id):
    """
    fetching popular user tags, more specific than the genre
    """
    url = f"https://steamspy.com/api.php?request=appdetails&appid={app_id}"
    try:
        response = requests.get(url, timeout=10) 
        
        if response.status_code == 200:
            data = response.json()
            tags_dict = data.get('tags', {})
            if isinstance(tags_dict, dict) and tags_dict:
                #--> extracting top 10 tage from the dict returned 
                top_tags = list(tags_dict.keys())[:10]
                return ", ".join(top_tags)
            else:
                return "No Tags" 
                
    except requests.exceptions.RequestException as e:
        return "server err"
        
    return "Failed" #if status is not 200

#--> collect tags 
extracted_tags = []

print(f"Starting API extraction for {len(df)} games")

#-->extraction loop
for index, row in tqdm(df.iterrows(), total=len(df), desc="Fetching Tags"):
    app_id = row['QueryID']
    
    #if the appID is null, tags is null
    if pd.isna(app_id):
        extracted_tags.append(np.nan)
        continue
        
    tag_string = get_top_tags(int(app_id))
    extracted_tags.append(tag_string)
    # avoidd api blocking
    time.sleep(1.5)

Starting API extraction for 11357 games


Fetching Tags: 100%|██████████| 11357/11357 [14:08:39<00:00,  4.48s/it]  ]


In [12]:
df['PopularUserTags'] = extracted_tags

In [14]:
df.to_csv('dataset_API.csv', index=False)

In [5]:
df = pd.read_csv('dataset_API.csv')

In [6]:
df

,QueryID,ResponseID,QueryName,ResponseName,ReleaseDate,RequiredAge,DemoCount,DeveloperCount,DLCCount,Metacritic,...,Reviews,SupportedLanguages,Website,PCMinReqsText,PCRecReqsText,LinuxMinReqsText,LinuxRecReqsText,MacMinReqsText,MacRecReqsText,PopularUserTags
0,10,10,Counter-Strike,Counter-Strike,Nov 1 2000,0,0,1,0,88,...,,English French German Italian Spanish Simplifi...,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"Action, FPS, Multiplayer, Shooter, Classic, Te..."
1,20,20,Team Fortress Classic,Team Fortress Classic,Apr 1 1999,0,0,1,0,0,...,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"Action, FPS, Multiplayer, Classic, Hero Shoote..."
2,30,30,Day of Defeat,Day of Defeat,May 1 2003,0,0,1,0,79,...,,English French German Italian Spanish,http://www.dayofdefeat.com/,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"FPS, World War II, Multiplayer, Shooter, Actio..."
3,40,40,Deathmatch Classic,Deathmatch Classic,Jun 1 2001,0,0,1,0,0,...,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"Action, FPS, Classic, Multiplayer, Shooter, Fi..."
4,50,50,Half-Life: Opposing Force,Half-Life: Opposing Force,Nov 1 1999,0,0,1,0,0,...,,English French German Korean,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"FPS, Action, Classic, Sci-fi, Singleplayer, Sh..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11352,567020,567020,TILE,TILE,Dec 9 2016,0,0,1,0,0,...,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) Vista / 7P...,,,,,,"Casual, Indie, Puzzle"
11353,567660,567660,Baseball Riot,Baseball Riot,Jan 17 2017,0,0,1,0,0,...,,English**languages with full audio support,http://www.10tons.com/Game/baseball_riot.html,Minimum:OS: Windows XP / Vista / 7 / 8 / 10Pro...,,,,,,"Casual, Indie, Sports, Action, Arcade, 2D, Sin..."
11354,567860,567860,Passage 4,Passage 4,Dec 13 2016,0,0,1,0,0,...,,English* French Italian German* Spanish Dutch*...,http://www.libredia.com,Minimum:OS: Windows 2000/XP/Vista/7/8/10Proces...,,,,,,"Clicker, Puzzle, Match 3, Board Game, Relaxing..."
11355,567940,567940,Piximalism,Piximalism,Sep 26 2019,0,0,1,0,0,...,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) XP / Vista...,Recommended:OS: Microsoft(r) Windows(r) XP / V...,,,,,"Action, Casual, Adventure, Indie, Pixel Graphics"


URLs info extraction

In [15]:
df['Has_Support_Link'] = df['SupportURL'].apply(lambda x: 0 if str(x).strip() == ""  else 1)

In [16]:
df['Has_Support_Link']

0        1
1        0
2        0
3        0
4        0
        ..
11352    0
11353    0
11354    0
11355    0
11356    0
Name: Has_Support_Link, Length: 11357, dtype: int64

In [18]:
#after url unique analysis in EDA
def categorize_support_tier(url):
    if pd.isna(url) or str(url).strip() == '':
        return 'None'

    url = str(url).lower()

    if any(keyword in url for keyword in ['support', 'help', 'ticket', 'custhelp', 'service', 'sqex', 'ubi']):
        return 'Professional_Portal'
    elif any(keyword in url for keyword in ['facebook', 'twitter', 'steam', 'forum', 'discord', 'reddit']):
        return 'Community_Social'

    else:
        return 'General_Website'

df['Support_Tier'] = df['SupportURL'].apply(categorize_support_tier)
print(df['Support_Tier'].value_counts())

Support_Tier
General_Website        4473
None                   4438
Professional_Portal    1735
Community_Social        711
Name: count, dtype: int64


In [21]:
output_file = 'image_metadata_11k.csv'

def fetch_dual_metadata(row_tuple):
    index, row = row_tuple
    app_id = str(row['QueryID'])
     #check both provided imgs
    urls = {
        'Header': row.get('HeaderImage'),
        'Background': row.get('Background')
    }
    
    result = {'QueryID': app_id}
    
    for key, url in urls.items():
        if pd.isna(url) or str(url).strip() == '':
            result[f'{key}_Size_Bytes'] = 0
            result[f'{key}_Exists'] = 0
            continue
            
        try:
            #Head --> fetches only the metadata no need to download the img to ram
            response = requests.head(url, timeout=5, allow_redirects=True)
            size = int(response.headers.get('Content-Length', 0))
            exists = 1 if response.status_code == 200 else 0
    
            result[f'{key}_Size_Bytes'] = size
            result[f'{key}_Exists'] = exists
        except:
            result[f'{key}_Size_Bytes'] = 0
            result[f'{key}_Exists'] = 0
            
    return result

rows_to_process = [r for r in df.iterrows()]
print(f"Starting metadata fetch for {len(rows_to_process)} remaining games...")
# parallelization use 10 threads to improve time
if rows_to_process:
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(fetch_dual_metadata, rows_to_process), 
                           total=len(rows_to_process), 
                           desc="Fetching Image Metadata"))

    new_metadata_df = pd.DataFrame([r for r in results if r is not None])
    
    # mode='a' adds to the file, header=not os.path.exists xxx duplicate
    new_metadata_df.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)
    print(f"Saved {len(new_metadata_df)} new rows to {output_file}")


full_metadata = pd.read_csv(output_file).drop_duplicates(subset=['QueryID'])
full_metadata['QueryID'] = full_metadata['QueryID'].astype(str)
df['QueryID'] = df['QueryID'].astype(str)

df = df.merge(full_metadata, on='QueryID', how='left')

Starting metadata fetch for 11357 remaining games...


Fetching Image Metadata: 100%|██████████| 11357/11357 [11:52<00:00, 15.95it/s]


Saved 11357 new rows to image_metadata_11k.csv


notice that most of these games back to 90's, so more size doesn't mean necessarily good quality

In [35]:
domain_counts = site_domains.value_counts()
top_entities = domain_counts[domain_counts >= 20].index.tolist() #--> if u need more change the head number

def game_web(url):
    if pd.isna(url) or str(url).strip() == '' or str(url).lower() == 'none':
        return 'None'
    domain = urlparse(str(url)).netloc.lower().replace('www.', '')
    if domain in [d.replace('www.', '') for d in top_entities]:
        if any(s in domain for s in ['facebook', 'twitter', 'steam', 'vk', 'indiedb']):
            return 'Has_Social_Only'
        return 'Major_Publisher'
    if any(s in domain for s in ['facebook', 'twitter', 'github', 'tumblr', 'wordpress']):
        return 'Has_Social_Only'
    return 'Custom_Studio'

df['Website_Authority'] = df['Website'].apply(game_web)

In [36]:
df

,QueryID,ResponseID,QueryName,ResponseName,ReleaseDate,RequiredAge,DemoCount,DeveloperCount,DLCCount,Metacritic,...,PopularUserTags,Has_Support_Link,Support_Tier,Header_Size_Bytes,Header_Exists,Background_Size_Bytes,Background_Exists,Header_Size_Group,ReleaseYear,Website_Authority
0,10,10,Counter-Strike,Counter-Strike,Nov 1 2000,0,0,1,0,88,...,"Action, FPS, Multiplayer, Shooter, Classic, Te...",1,Community_Social,28138,1,50690,1,Small,2000.0,None
1,20,20,Team Fortress Classic,Team Fortress Classic,Apr 1 1999,0,0,1,0,0,...,"Action, FPS, Multiplayer, Classic, Hero Shoote...",0,None,30412,1,64709,1,Small,1999.0,None
2,30,30,Day of Defeat,Day of Defeat,May 1 2003,0,0,1,0,79,...,"FPS, World War II, Multiplayer, Shooter, Actio...",0,None,36605,1,58479,1,Medium,2003.0,Custom_Studio
3,40,40,Deathmatch Classic,Deathmatch Classic,Jun 1 2001,0,0,1,0,0,...,"Action, FPS, Classic, Multiplayer, Shooter, Fi...",0,None,32228,1,39026,1,Small,2001.0,None
4,50,50,Half-Life: Opposing Force,Half-Life: Opposing Force,Nov 1 1999,0,0,1,0,0,...,"FPS, Action, Classic, Sci-fi, Singleplayer, Sh...",0,None,41595,1,37607,1,Medium,1999.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11352,567020,567020,TILE,TILE,Dec 9 2016,0,0,1,0,0,...,"Casual, Indie, Puzzle",0,None,10980,1,146,0,Small,2016.0,None
11353,567660,567660,Baseball Riot,Baseball Riot,Jan 17 2017,0,0,1,0,0,...,"Casual, Indie, Sports, Action, Arcade, 2D, Sin...",0,None,56763,1,48499,1,Large,2017.0,Custom_Studio
11354,567860,567860,Passage 4,Passage 4,Dec 13 2016,0,0,1,0,0,...,"Clicker, Puzzle, Match 3, Board Game, Relaxing...",0,None,63687,1,116959,1,Extra_Large,2016.0,Major_Publisher
11355,567940,567940,Piximalism,Piximalism,Sep 26 2019,0,0,1,0,0,...,"Action, Casual, Adventure, Indie, Pixel Graphics",0,None,28107,1,146,0,Small,2019.0,None


In [41]:
df.to_csv('data_urls_final.csv', index=False)